# Training loop for mini-gLM

## Load data

Import packages and configure root.

In [ ]:
%pip install pyfaidx
%pip install wandb
%pip install yaml

import numpy as np
import pandas as pd
import sys, subprocess
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
import yaml

from pathlib import Path
from huggingface_hub import hf_hub_download
from torch.utils.data import DataLoader

# Configure root
COLAB = Path("/content").exists()
repo_url = "https://github.com/eddykang06/mini-gLM.git"
repo_dir = Path("mini-gLM")
if COLAB:
    root = Path("/content/mini-gLM")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", repo_url])
else:
    root = Path.cwd().parent
sys.path.insert(0, str(root))

# Use GPU if available
torch.manual_seed(111)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Training

Configure parameters.

In [ ]:
from src.data import get_pretraining_data

# Get train and val data
train_dataset, val_dataset = get_pretraining_data(root)

# Define model params
model_params = {
    "vocab_size": 512,
    "num_blocks": 5, 
    "d_model": 384,     
    "num_heads": 8,  
    "h_dim": 1024,      
    "num_experts": 8,
    "top_k": 2,
    "p_drop": 0.1
}

Initialize wandb run.

In [ ]:
run = wandb.init(
    entity = "eddykang06-harvard-university",
    project = "glm",

    # Store hyperparameters
    config = {
        
    }
)

Train.

In [ ]:
from src.train import train_dense_glm

# Training loop
model = train_dense_glm(
    max_epochs = 30,
    lr = 0.0001,
    model_params = model_params,
    weight_decay = 0.01,
    batch_space = 32000,
    predict_prob = 0.15,
    masking_prob = 0.80,
    mutate_prob = 0.10,
    train_dataset = train_dataset,
    val_dataset = val_dataset,
    device = device,
    val_every = 1500,
    run = run
)